In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --- File path ---
file_path = "DATA/FRD_FILES/g101935241.avp"

# --- Find where data begins ---
with open(file_path) as f:
    lines = f.readlines()
start_idx = next((i for i, line in enumerate(lines) if line.startswith("AVAPS-D01")), None)

# --- Read from first data line onward ---
df = pd.read_csv(
    file_path,
    sep=r"\s+",
    skiprows=start_idx,
    names=[
        "Type", "Code", "ID", "Date", "Time",
        "Press_mb", "Temp_C", "RH_pct", "Wind_Dir_deg", "Wind_Spd_mps",
        "Vert_Vel_mps", "Lon_deg", "Lat_deg", "Alt_m",
        "GPS_Wnd", "RH1", "RH2", "Sonde_Sat", "Error", "Alt2_m"
    ]
)

# --- Convert numeric columns safely ---
cols_to_num = ["Press_mb", "Temp_C", "RH_pct", "Wind_Dir_deg", "Wind_Spd_mps", "Alt_m"]
for col in cols_to_num:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# --- Drop NaNs ---
df = df.dropna(subset=cols_to_num)

# --- Remove physically impossible / flagged values ---
df = df[
    (df["Press_mb"].between(100, 1100)) &
    (df["Temp_C"].between(-90, 60)) &
    (df["RH_pct"].between(0, 105)) &
    (df["Wind_Spd_mps"].between(0, 120)) &
    (df["Wind_Dir_deg"].between(0, 360)) &
    (df["Alt_m"].between(0, 50000))
]

# --- Sort by altitude ---
df = df.sort_values("Alt_m")

# --- Check cleaned data ---
print("\n✅ Final cleaned data preview (first 15 rows):\n")
print(df[["Alt_m", "Press_mb", "Temp_C", "RH_pct", "Wind_Spd_mps", "Wind_Dir_deg"]].head(15).to_string(index=False))

print("\nSummary of realistic ranges:")
for col in cols_to_num:
    print(f"{col}: {df[col].min():.2f} – {df[col].max():.2f}")

# --- Plot temperature profile ---
plt.figure(figsize=(5, 7))
plt.plot(df["Temp_C"], df["Alt_m"], color="firebrick")
plt.xlabel("Temperature (°C)")
plt.ylabel("Altitude (m)")
plt.title("Temperature Profile (Cleaned)")
plt.grid(True)
plt.show()

# --- Plot RH profile ---
plt.figure(figsize=(5, 7))
plt.plot(df["RH_pct"], df["Alt_m"], color="teal")
plt.xlabel("Relative Humidity (%)")
plt.ylabel("Altitude (m)")
plt.title("Relative Humidity Profile (Cleaned)")
plt.grid(True)
plt.show()

# --- Plot wind speed & direction ---
fig, ax1 = plt.subplots(figsize=(5, 7))
ax1.plot(df["Wind_Spd_mps"], df["Alt_m"], color="purple", label="Wind Speed (m/s)")
ax1.set_xlabel("Wind Speed (m/s)")
ax1.set_ylabel("Altitude (m)")
ax1.grid(True)

ax2 = ax1.twiny()
ax2.plot(df["Wind_Dir_deg"], df["Alt_m"], color="orange", label="Wind Direction (°)")
ax2.set_xlabel("Wind Direction (°)")

fig.suptitle("Wind Speed & Direction (Cleaned)")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

file_path = "DATA/FRD_FILES/g101935241.frd"

# --- Read file lines ---
with open(file_path) as f:
    lines = f.readlines()

# Find the header line
header_idx = [i for i, line in enumerate(lines) if line.strip().startswith("IX")][0]

# Read data starting just after header
df = pd.read_csv(
    file_path,
    sep='\s+',
    skiprows=header_idx+1,
    header=None,
    na_values=[-999, -999.0, -9999, -9999.0]
)

# Assign proper column names
columns = [
    "IX", "t_s", "P_mb", "T_C", "RH_pct", "Z_m", "WD_deg", "WS_mps",
    "U_mps", "V_mps", "NS", "WZ_mps", "ZW_m", "FP", "FT", "FH", "FW",
    "LAT_deg", "LON_deg"
]
df.columns = columns

# Convert key columns to numeric
for col in ["Z_m","T_C","RH_pct","WS_mps","WD_deg","LAT_deg","LON_deg"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop invalid rows
df = df.dropna(subset=["Z_m","T_C","RH_pct","WS_mps"])

# --- Plot Temperature vs Altitude ---
plt.figure(figsize=(5,7))
plt.plot(df["T_C"], df["Z_m"], color="red")
plt.xlabel("Temperature (°C)")
plt.ylabel("Altitude (m)")
plt.title("Temperature Profile")
plt.grid(True)
plt.show()

# --- Plot RH vs Altitude ---
plt.figure(figsize=(5,7))
plt.plot(df["RH_pct"], df["Z_m"], color="blue")
plt.xlabel("Relative Humidity (%)")
plt.ylabel("Altitude (m)")
plt.title("RH Profile")
plt.grid(True)
plt.show()

# --- Plot Wind Speed vs Altitude ---
plt.figure(figsize=(5,7))
plt.plot(df["WS_mps"], df["Z_m"], color="green")
plt.xlabel("Wind Speed (m/s)")
plt.ylabel("Altitude (m)")
plt.title("Wind Speed Profile")
plt.grid(True)
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(6,8))

# --- Temperature ---
ax1.plot(df["T_C"], df["Z_m"], color="red", label="Temperature (°C)")
ax1.set_xlabel("Temperature (°C) / RH (%) / Wind Speed (m/s)")
ax1.set_ylabel("Altitude (m)")
ax1.grid(True)

# --- Relative Humidity ---
ax2 = ax1.twiny()
ax2.plot(df["RH_pct"], df["Z_m"], color="blue", label="RH (%)")
ax2.set_xlabel("Relative Humidity (%)")

# --- Wind Speed ---
ax3 = ax1.twiny()
ax3.spines["top"].set_position(("axes", 1.1))  # move the third x-axis above
ax3.plot(df["WS_mps"], df["Z_m"], color="green", label="Wind Speed (m/s)")
ax3.set_xlabel("Wind Speed (m/s)")

plt.title("Dropwindsonde Profiles")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Fixed list of columns for the data part
DATA_COLUMNS = ['IX','t','P','T','RH','Z','WD','WS','U','V','NS','WZ','ZW','FP','FT','FH','FW','LAT','LON']

def read_frd(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Find the header line starting with 'IX'
    header_idx = next(i for i, line in enumerate(lines) if line.startswith('IX'))

    # Data lines are everything after the header
    data_lines = lines[header_idx+1:]
    data = [line.split() for line in data_lines if line.strip()]  # ignore empty lines

    # Create DataFrame with correct number of columns
    df = pd.DataFrame(data, columns=DATA_COLUMNS)

    # Convert numeric columns and replace -999 with NaN
    for col in ['t','P','T','RH','Z','WD','WS','U','V','WZ','ZW','LAT','LON']:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace(-999, pd.NA)

    df = df.dropna(subset=['T','Z'])
    return df

# --- Loop through all FRD files from CSV ---
file_list = pd.read_csv("hurricane_edouard_files.csv")
for idx, row in file_list.iterrows():
    frd_path = Path(row['filepath'])

    if not frd_path.exists():
        print(f"File not found: {frd_path}")
        continue

    df = read_frd(frd_path)
    if df.empty:
        print(f"No valid data in {frd_path}")
        continue

    # Plot Temperature vs Altitude
    plt.figure(figsize=(5,7))
    plt.plot(df['T'], df['Z'], marker='o', markersize=2)
    plt.xlabel("Temperature (°C)")
    plt.ylabel("Altitude (m)")
    plt.ylim(bottom=0)  # 0 m at bottom
    plt.grid(True)
    plt.title(f"{frd_path.name} - Temp vs Altitude")
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Fixed list of columns for the data part
DATA_COLUMNS = ['IX','t','P','T','RH','Z','WD','WS','U','V','NS','WZ','ZW','FP','FT','FH','FW','LAT','LON']

def read_frd(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Find the header line starting with 'IX'
    header_idx = next(i for i, line in enumerate(lines) if line.startswith('IX'))

    # Data lines are everything after the header
    data_lines = lines[header_idx+1:]
    data = [line.split() for line in data_lines if line.strip()]  # ignore empty lines

    # Create DataFrame with correct number of columns
    df = pd.DataFrame(data, columns=DATA_COLUMNS)

    # Convert numeric columns and replace -999 with NaN
    for col in ['t','P','T','RH','Z','WD','WS','U','V','WZ','ZW','LAT','LON']:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace(-999, pd.NA)

    df = df.dropna(subset=['T','Z'])
    return df

# Output folder for PNGs
output_dir = Path("plots")
output_dir.mkdir(exist_ok=True)

# --- Loop through all FRD files from CSV ---
file_list = pd.read_csv("hurricane_edouard_files.csv")
for idx, row in file_list.iterrows():
    frd_path = Path(row['filepath'])

    if not frd_path.exists():
        print(f"File not found: {frd_path}")
        continue

    df = read_frd(frd_path)
    if df.empty:
        print(f"No valid data in {frd_path}")
        continue

    # Plot Temperature vs Altitude
    plt.figure(figsize=(5,7))
    plt.plot(df['T'], df['Z'], marker='o', markersize=2)
    plt.xlabel("Temperature (°C)")
    plt.ylabel("Altitude (m)")
    plt.ylim(bottom=0)  # 0 m at bottom
    plt.grid(True)
    plt.title(f"{frd_path.name} - Temp vs Altitude")
    plt.tight_layout()
    
    # Save figure as PNG
    save_path = output_dir / f"{frd_path.stem}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()  # Close the figure to free memory
    print(f"Saved plot: {save_path}")


In [ ]:
import cv2
from pathlib import Path
import shutil  # for moving files

# Folder containing your PNGs
img_dir = Path("plots")
img_files = sorted(img_dir.glob("*.png"))  # simple alphabetical sort

if not img_files:
    raise ValueError("No PNG files found in the folder!")

# Read the first image to get the frame size
first_img = cv2.imread(str(img_files[0]))
height, width, layers = first_img.shape
size = (width, height)

# --- Step 1: Save video locally first ---
local_video = Path("temperature_vs_altitude.mp4")

fps = 2  # frames per second
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(str(local_video), fourcc, fps, size)

for img_file in img_files:
    img = cv2.imread(str(img_file))
    out.write(img)

out.release()
print(f"Saved local video: {local_video}")
